# PE6201 · A2 — scaffold tour · **PROBLEM A**

## What this notebook is

**One case, followed end to end, with real values printed at every step.**

The case is **`CLM-8842`** — a Problem A claim, and the partly-payable worked example
in Appendix A of the brief. Every cell below uses that one claim. Wherever you see a
hard-coded value — `'CLM-8842'`, `'M-2214'`, `'POL-3310'`, a procedure code — it came
from that record, and the markdown says where.

**The model is simulated, the data is real.** The backend replays a fixed sequence of
moves (`SCRIPTS['CLM-8842']` in `backends.py`) so the run is deterministic and free.
The tools underneath are doing genuine lookups against the shipped JSON — nothing is
faked except the model's decisions.

> **Doing Problem B?** Use `A2_Scaffold_Tour_ProblemB.ipynb` instead. Same nine steps,
> same scaffold, different case. You only need the one for the problem you chose.

## What this notebook is not

**It contains no logic of its own.** Every cell imports from the `.py` files beside it.

> **Notebooks explore. Modules ship.**

Six people can edit six modules at once. Six people editing one notebook produces merge
conflicts and an unreadable diff — and section 8 of the brief leans on your commit
history to corroborate who did what.

**What you submit is the modules and `run_eval.py`, not this notebook.** D5(a) says a
marker clones your repository and runs it. `python3 run_eval.py` is that; a notebook
is not.

---
### Running this in Colab
Upload `A2_scaffold/` and `A2_reference_data/` to the same folder in your Drive, then
edit the two paths in the setup cell. Running locally: just run the cells in order.


## Cell 1 · Setup

Puts the scaffold on the import path, **switches to Problem A**, and prints which
backend is active.

`config.PROBLEM = 'A'` is set here so the notebook works whatever the file says. When
you commit, set it in `config.py` itself rather than relying on a notebook cell.

**Read the printed line.** `BACKEND=scripted` is what makes this free, offline and
identical every time — and it must be the committed default in your submission.

**If the printed line disagrees with `config.py`**, the scaffold will say so in capitals — Python is reusing cached bytecode. Restart the kernel and re-run.


In [ ]:
# --- SETUP ---------------------------------------------------------
# Local: this cell works as-is. Colab: set the two paths below.
import os, sys, json

SCAFFOLD = '.'          # e.g. '/content/drive/MyDrive/PE6201/A2_scaffold'
# os.environ['A2_DATA'] = '/content/drive/MyDrive/PE6201/A2_reference_data'

sys.path.insert(0, SCAFFOLD)
import config
config.PROBLEM = 'A'            # this notebook is Problem A

print(config.summary())
print('data:', config.data_root())


---
## Cell 2 · What the agent is handed

The harness gives the agent **one claim id and nothing else**. This cell fetches
`CLM-8842` so you can see exactly what that record contains.

**Look at what is *not* in it.** No policy. No coverage verdict. No panel status. No
pre-authorisation. Every one of those has to be fetched by a tool during the run.

**Three fields drive everything below:**

- `member_id` is **`M-2214`** — the route to the policy, and half the duplicate check.
- `hospital_id` is **`H-114`** — panel status.
- `lines` has **three entries**. That is the number that matters most: **every line
  needs its own coverage check and its own disposition.** Nine of the fifteen shipped
  claims have one line; six have two to four. An agent that checks only the first line
  quietly approves things it should refuse.


In [ ]:
import tools

claim = tools.get_claim('CLM-8842')
print(json.dumps(claim, indent=1))
print()
print('member   :', claim['member_id'])
print('hospital :', claim['hospital_id'])
print('lines    :', len(claim['lines']), '->',
      [l['code'] for l in claim['lines']])


---
## Cell 3 · The policy — where most refusals come from

`lookup_policy('M-2214')` — **`'M-2214'` because that is this claim's member**, from
cell 2 — follows claim → member → policy. Two hops, and the first carries no decision
information at all.

**Three separate escalation reasons live in the row it returns**, and they are easy to
conflate:

1. `status == 'lapsed'` — nothing else matters
2. date of service outside `start_date .. end_date` — **even if status says active**
3. the lines together exceed `remaining`

**`remaining` is `annual_limit` minus `used_to_date`**, and the claim total is tested
against *that*, not against the limit. Testing against the limit is a silent wrong
answer on any policy with spend on it.

For this claim: active, in dates, and the total is well under the headroom. So none of
the three fires and the run continues.


In [ ]:
pol = tools.lookup_policy('M-2214')          # M-2214: see cell 2
p = pol['policy']
print('policy      :', p['policy_id'], p['product'])
print('status      :', p['status'])
print('cover dates :', p['start_date'], '..', p['end_date'],
      '   claim date:', claim['date_of_service'])
print('annual limit: %5d' % p['annual_limit'])
print('used to date: %5d' % p['used_to_date'])
print('REMAINING   : %5d   <- the claim is tested against THIS' % pol['remaining'])
print()
total = sum(l['amount'] for l in claim['lines'])
print('claim total : %5d  -> %s' % (total,
      'inside the headroom' if total <= pol['remaining'] else 'OVER - escalate'))
print()
print('exclusions on this policy:', json.dumps(p['exclusions']))


---
## Cell 4 · The three lines — and the two fields that drive the run

`check_coverage` is called **once per line**. Three lines, three calls — and because
they are independent of each other, all three belong in the same turn.

The table below shows the two fields that decide what happens next:

**`requires_preauth` is THE BRANCH.** Only `62480` needs one. An agent that calls
`get_preauthorisation` three times has not read the flag — and a claim where no line
needs one finishes a whole turn earlier. *That* is why turn counts vary by claim: you
did not write the branch, the record did.

**`excluded` refuses THE LINE, not the claim.** `31255` is excluded under EX-14, but
the decision is still `approve_in_principle` — three lines approved-or-refused in **one
decision letter covering both**. Escalating the whole claim because one line is
excluded is a distinct and common failure, and the record must name the rule, not just
say 'excluded'.


In [ ]:
print('%-8s %-30s %-16s %-9s %s'
      % ('code', 'description', 'requires_preauth', 'excluded', 'rule'))
print('-' * 84)
for line in claim['lines']:
    cov = tools.check_coverage(line['code'], 'POL-3310')   # POL-3310: see cell 3
    print('%-8s %-30s %-16s %-9s %s'
          % (cov['code'], cov['description'][:30],
             cov['requires_preauth'], cov['excluded'],
             cov['exclusion_rule'] or ''))
print()
print('-> only ONE line needs a pre-authorisation, so ONE call, not three')
print('-> one line is excluded: it is REFUSED, the claim is not')


---
## Cell 5 · The trap in this case

This claim is **not** a duplicate. But the claims history contains a decided claim on
**the same date of service**, for a different member at a different hospital.

So an agent matching duplicates on anything less than **all four facts** — member,
hospital, date of service, lines — **wrongly escalates this perfectly good claim.**

The cell below runs four matching strategies against the whole queue and shows which
claims each one flags. `CLM-8933` is the **one** true duplicate; anything else in a row
is a false positive — a good claim wrongly escalated.

**Only the last strategy is correct.** Date-only matching wrongly flags *this* claim.
The other two shortcuts do not touch `CLM-8842`, but they wrongly flag `CLM-8960`
instead — so all three shortcuts fail, just on different claims. The cell prints which,
so you do not have to take that on trust.

The three near-misses in the history are there deliberately, so that a shortcut match
is punished rather than getting lucky.

*(This cell reaches into `tools._load` to show you the raw table. That is a teaching
shortcut — your agent must never do it.)*


In [ ]:
claims  = tools._load('A', 'claims')
decided = tools._load('A', 'decided_claims')
norm = lambda ls: sorted((x['code'], x['amount']) for x in ls)

strategies = {
    'date only            ': lambda c, d: c['date_of_service'] == d['date_of_service'],
    'member + date        ': lambda c, d: (c['member_id'], c['date_of_service'])
                                          == (d['member_id'], d['date_of_service']),
    'member+hospital+date ': lambda c, d: (c['member_id'], c['hospital_id'],
                                           c['date_of_service'])
                                          == (d['member_id'], d['hospital_id'],
                                              d['date_of_service']),
    'ALL FOUR FACTS       ': lambda c, d: (c['member_id'], c['hospital_id'],
                                           c['date_of_service']) ==
                                          (d['member_id'], d['hospital_id'],
                                           d['date_of_service'])
                                          and norm(c['lines']) == norm(d['lines']),
}

for label, match in strategies.items():
    hits = sorted({c['claim_id'] for c in claims for d in decided if match(c, d)})
    ok = (hits == ['CLM-8933'])
    flag = 'CORRECT' if ok else 'WRONG - false positives'
    print('%s flags %-40s %s' % (label, ', '.join(hits), flag))
print()
# Computed, not asserted - so this notebook cannot drift from the data.
this = 'CLM-8842'
wrong_here = [lbl.strip() for lbl, m in strategies.items()
              if any(m(c, d) for c in claims if c['claim_id'] == this for d in decided)]
wrong_anywhere = [lbl.strip() for lbl, m in strategies.items()
                  if sorted({c['claim_id'] for c in claims for d in decided
                             if m(c, d)}) != ['CLM-8933']]
print('%s - the claim in this notebook - is NOT a duplicate.' % this)
print('  strategies that wrongly flag THIS claim   : %s' % (', '.join(wrong_here) or 'none'))
print('  strategies that wrongly flag SOME claim   : %s' % ', '.join(wrong_anywhere))
print()
print('Every shortcut fails somewhere. Only the full four-fact match is right.')


---
## Cell 6 · The run itself

Now the whole thing, turn by turn. `verbose=True` prints the model's thought and every
tool result.

**Eight tool calls, four turns** — and this matches Appendix A exactly, which is worth
checking rather than taking on trust:

| Turn | Calls | Why grouped this way |
|---|---|---|
| 1 | `get_claim` | must run alone — everything else needs the member, hospital and lines |
| 2 | `lookup_policy` + `check_coverage` ×3 + `lookup_hospital` | **five calls**, all independent |
| 3 | `get_preauthorisation` | **cannot** join turn 2 — you do not know which line needs one until coverage answers |
| 4 | `issue_decision_letter` | the gated action — a turn like any other |

**Turn 3 is the dependency rule made visible.** The three coverage checks fold into one
turn because they are independent. The pre-authorisation cannot, because it depends on
their answer. Run all eight sequentially and it is eight turns; fold the five and it is
four — a 54% token saving, and D2(c) asks you to reason about exactly this.

The `thought` text is scripted, not generated. It is there to show what a model *would*
be reasoning at each step.


In [ ]:
from agent import run_case

record = run_case('CLM-8842', problem='A', verbose=True)


---
## Cell 7 · The decision record

This is what gets graded. Two halves worth separating:

**The answer** — `decision` and `reason`. Note the reason carries a disposition for
*every* line, both totals, and the exclusion rule by name.

**The instrumentation** — `turns`, `tokens_in`, `cost_usd`, `evidence`,
`guardrails_fired`. Captured *while the run happened*, because you cannot report a
failure you had no way of noticing. D6's cost model and D7's loop failure both need
these.

Note `guardrails_fired` shows `gate_passed` for `issue_decision_letter` — the
irreversible step went through the autonomy gate, and the record proves it.

**The token numbers are estimates**, because the scripted backend has no model. D6
wants *measured* counts, which means the live battery.


In [ ]:
print(json.dumps(record, indent=2))


---
## Cell 8 · Grading — the code check

Deterministic comparison against the answer key. **No model, no person, no opinion.**
This is what produces the number.

The key row for `CLM-8842` is printed first. Note it has no `booked` field — that is
Problem B only; Problem A never books anything.

What is **not** compared: the wording, the turn count, the cost. Two agents can both be
right and cost very different amounts, which is the subject of D6.


In [ ]:
from harness import load_key, code_check, prepare_judgement_check

expected = load_key('A')['CLM-8842']
print('THE ANSWER KEY SAYS:')
print(json.dumps(expected, indent=1))

passed, fails = code_check(record, expected)
print()
print('CODE CHECK:', 'PASS' if passed else 'FAIL')
for f in fails:
    print('   ', f)


---
## Cell 9 · Grading — the judgement check

**This is the half a pass rate cannot show you.** With three possible outcomes a
coin-flip scores 33% on the code check alone, and an agent can reach the right decision
for the wrong reason without the code check noticing.

For this claim `must_record` asks for five things — including *a disposition for all 3
lines* and the exclusion that caught `31255`. An agent that said only "approved" would
pass the code check and fail here, which is exactly the point.

`prepare_judgement_check` **builds a queue, it does not decide**. Someone reads the
reason and rules on each item: a person can, or a second model can. Same kind of check;
the only difference is who grades. If you automate it with a model, say so in the
report — a model grading a model is a claim that needs defending.


In [ ]:
item = prepare_judgement_check(record, expected)

print('The reason the agent gave:')
print('   ', item['reason'])
print()
print('Does it carry each of these? Nobody has ruled yet:')
for m in item['must_record']:
    print('  [ ]', m)
print()
print('verdict:', item['verdict'], '   graded_by:', item['graded_by'])


---
## Cell 10 · The failure that raises no exception  (D7)

Same case, same data. **One guard deleted** — action de-duplication — and nothing else
changed. That is the shape D7 requires: *the working agent, minus X*. Putting X back
recovers the behaviour, which is what makes it a diagnosis rather than a story.

Watch the three numbers in the BEFORE and AFTER lines:

- **turns** 4 → 6
- **cost** roughly 1.8× higher
- **decision** unchanged — still `approve_in_principle`

**No exception. No error. The right answer.** A pass-rate table would show this run as
a clean pass. And neither the step cap (8 turns) nor the budget ceiling (60,000 tokens)
fires, because neither is breached — they bound the damage, they do not detect the
fault.

You only ever see this **if you are counting**. That is the whole lesson of D7.


In [ ]:
import demo_loop_failure
demo_loop_failure.main(case='CLM-8842', problem='A')


---
## Cell 11 · Now make it yours

Everything above ran on **one scripted case**. Your set needs 30–50.

1. **`config.py`** — set `PROBLEM = 'A'` in the file, not just in this notebook.
2. **`tools.py`** — read the comment block on every tool, then the six-field
   descriptors at the bottom. Run **`python3 run_eval.py --prompt`** to see how those
   descriptors become the text the model is sent. Now write deliberately worse ones:
   that is your D2(b) **v1**, and the measured comparison is the deliverable.
3. **`backends.py`** — script a second claim yourself. **If you cannot write the steps
   down, you do not yet understand the case.** Better to find that out now than at 2am
   on the 13th. `CLM-8933` — the duplicate — is a good second one.
4. **`expected_outcomes_A.json`** — label every case you add, **from Appendix A's
   routing table, before you run the agent on it**. See
   `PE6201_A2_Adding_Extra_Cases.pdf`.
5. **`guardrails.py`** — set the limits from evidence. If your median run is 4 turns
   and your worst legitimate run is 7, a cap of 8 is defensible and a cap of 30 is
   decoration.

Then close this notebook and work in the modules.

```bash
python3 run_eval.py
```
